# Prompt

How do I use geopandas?

# Prompt

How do I install geopandas directly in a Jupyter Notebook?

In [ ]:
# %conda install geopandas -y

In [ ]:
import geopandas as gpd

In [ ]:
import pandas as pd


def read_gbif_export(file_path, sep="\t", encoding="utf-8"):
    """
    Read a GBIF occurrence export into a pandas DataFrame.

    Parameters
    ----------
    file_path : str
        Path to the GBIF export file.
    sep : str, optional
        Field delimiter. GBIF downloads are usually tab-delimited ('\\t').
    encoding : str, optional
        File encoding, usually 'utf-8'.

    Returns
    -------
    pandas.DataFrame
        DataFrame containing the GBIF records.
    """
    df = pd.read_csv(
        file_path,
        sep=sep,
        encoding=encoding,
        low_memory=False
    )

    return df

df = read_gbif_export("data/0006970-260623161305970.csv")
df.head(2)

In [ ]:
def read_gadm_gpkg(file_path, layer=None):
    """
    Read a GADM GeoPackage into a GeoDataFrame.

    Parameters
    ----------
    file_path : str
        Path to the .gpkg file.
    layer : str, optional
        Layer name to read. If None, the first layer is loaded.

    Returns
    -------
    geopandas.GeoDataFrame
        GeoDataFrame containing the GADM boundaries.
    """
    gdf = gpd.read_file(file_path, layer=layer)

    return gdf

gdf = read_gadm_gpkg("data/maine_gadm_410.gpkg")
gdf.head(2)

# Prompt

I have a dataframe using Darwin Core that contains coordinates. I have a geopackage file from GADM that contains reference polygons. Please write a function to identify coordinates in the Darwin Core dataframe that plot outside the reference polygons.

In [ ]:
from shapely.geometry import Point


def find_coordinates_outside_polygons(
    dwc_df,
    gadm_gdf,
    lat_col="decimalLatitude",
    lon_col="decimalLongitude"
):
    """
    Identify Darwin Core records whose coordinates fall outside
    a set of reference polygons.

    Parameters
    ----------
    dwc_df : pandas.DataFrame
        Darwin Core dataframe.
    gadm_gdf : geopandas.GeoDataFrame
        GADM polygons.
    lat_col : str, default='decimalLatitude'
        Latitude column name.
    lon_col : str, default='decimalLongitude'
        Longitude column name.

    Returns
    -------
    pandas.DataFrame
        Subset of the original dataframe containing records
        outside the reference polygons.
    """

    # Keep only rows with valid coordinates
    valid = dwc_df.dropna(subset=[lat_col, lon_col]).copy()

    # Convert coordinates to point geometries
    points = gpd.GeoDataFrame(
        valid,
        geometry=gpd.points_from_xy(
            valid[lon_col],
            valid[lat_col]
        ),
        crs="EPSG:4326"
    )

    # Reproject points if necessary
    if points.crs != gadm_gdf.crs:
        points = points.to_crs(gadm_gdf.crs)

    # Spatial join
    joined = gpd.sjoin(
        points,
        gadm_gdf[["geometry"]],
        how="left",
        predicate="within"
    )

    # Points that matched no polygon
    outside = joined[joined["index_right"].isna()]

    # Return original columns (without spatial join fields)
    return dwc_df.loc[outside.index]

find_coordinates_outside_polygons(df, gdf)